In [ ]:
import altair as alt
import pandas as pd
from vega_datasets import data

alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

In [70]:
PAGE_BG, INK, INK_SOFT, GRID = "#faf8f1", "#1a1a17", "#4a4a44", "#e9e6df"
GAIN, LOSS = "#1b7f79", "#c2562b"

GAIN_TINT, LOSS_TINT = "#d6ece9", "#f6ddd1"
NEUTRAL_TINT, STABLE_TINT = "#f3e9c9", "#e9e6df"

REGION = {"Northeast": "#4c6a92", "Midwest": "#d4a03a",
          "South": "#8e5c8a", "West": "#7a9a6a"}
URBANICITY = {"Rural": "#b2e2e2", "Small metro": "#66c2a4",
              "Mid-size metro": "#2ca25f", "Large metro": "#006d2c"}

def seq(color):
    """Sequential scale where 0 = page background."""
    return alt.Scale(range=[PAGE_BG, color], interpolate="lab", domain = [0.15,0.5])

def diverging(m):
    """Symmetric diverging scale: loss (rust) -> background -> gain (teal)."""
    return alt.Scale(domain=[-m, 0, m], range=[LOSS, PAGE_BG, GAIN],
                     interpolate="lab")

@alt.theme.register("project", enable=True)   
def project_theme():
    return {"config": {
        "background": PAGE_BG,
        "view": {"stroke": None},
        "title": {"color": INK, "fontSize": 16, "subtitleColor": INK_SOFT, "anchor": "start"},
        "axis": {"labelColor": INK_SOFT, "titleColor": INK_SOFT,
                 "gridColor": GRID, "domainColor": INK_SOFT, "tickColor": INK_SOFT},
        "legend": {"labelColor": INK_SOFT, "titleColor": INK_SOFT},
    }}

In [ ]:
#Download Migration Patterns by Race and Parental Income at opportunityinsights.org/data
race_inc = pd.read_csv('../data/od.csv')
agg = pd.read_csv('../data/od_pooled.csv')

race_inc[["race", "income_quintile"]] = race_inc["pool"].str.extract(r"^(.*?)Q(\d)$")

In [4]:
#add census regions
divisions = {
    "New England":        ["Connecticut", "Maine", "Massachusetts", "New Hampshire",
                           "Rhode Island", "Vermont"],
    "Middle Atlantic":    ["New Jersey", "New York", "Pennsylvania"],
    "East North Central": ["Illinois", "Indiana", "Michigan", "Ohio", "Wisconsin"],
    "West North Central": ["Iowa", "Kansas", "Minnesota", "Missouri", "Nebraska",
                           "North Dakota", "South Dakota"],
    "South Atlantic":     ["Delaware", "District of Columbia", "Florida", "Georgia", "Maryland",
                           "North Carolina", "South Carolina", "Virginia", "West Virginia"],
    "East South Central": ["Alabama", "Kentucky", "Mississippi", "Tennessee"],
    "West South Central": ["Arkansas", "Louisiana", "Oklahoma", "Texas"],
    "Mountain":           ["Arizona", "Colorado", "Idaho", "Montana", "Nevada", "New Mexico",
                           "Utah", "Wyoming"],
    "Pacific":            ["Alaska", "California", "Hawaii", "Oregon", "Washington"],
}

division_to_region = {
    "New England": "Northeast", "Middle Atlantic": "Northeast",
    "East North Central": "Midwest", "West North Central": "Midwest",
    "South Atlantic": "South", "East South Central": "South", "West South Central": "South",
    "Mountain": "West", "Pacific": "West",
}

state_to_division = {s: d for d, states in divisions.items() for s in states}
state_to_region = {s: division_to_region[d] for s, d in state_to_division.items()}

In [5]:
#add regions to long race income
for side in ["o", "d"]:
    race_inc[f"{side}_division"] = race_inc[f"{side}_state_name"].map(state_to_division)
    race_inc[f"{side}_region"] = race_inc[f"{side}_state_name"].map(state_to_region)

In [7]:
#bring in census data to map commuting zones to counties, urban/rural
#based on 1990 mapping (to align with cohort)

cz_map = pd.read_excel('../data/czlma903.xls', dtype={"County FIPS Code": str})
cz_map = cz_map.rename(columns={"CZ90": "cz", "Population 1990": "pop",
                        "Rural-urban Continuum Code 1993 (Beale Code)": "beale"})
cz_map['beale'] = cz_map['beale'].replace('.', 9).astype(int)
cz_map["metro_pop"] = cz_map["pop"].where(cz_map["beale"] <= 3, 0)
cz_map['county_metro'] = cz_map['beale'] <= 3

cz_attrs = (cz_map.groupby("cz", as_index=False)
              .agg(cz_pop=("pop", "sum"), metro_pop=("metro_pop", "sum")))
cz_attrs["pct_metro_pop"] = cz_attrs["metro_pop"] / cz_attrs["cz_pop"]

#assign urban labels based on population
def urbanicity(r):
    if r["pct_metro_pop"] < 0.5:
        return "Rural"
    if r["cz_pop"] < 250_000:
        return "Small metro"
    if r["cz_pop"] < 1_000_000:
        return "Mid-size metro"
    return "Large metro"

cz_attrs["urbanicity"] = cz_attrs.apply(urbanicity, axis=1)

cz_map = cz_map.merge(
    cz_attrs[['cz', 'cz_pop', 'pct_metro_pop', 'urbanicity']], on='cz', how = 'left'
).drop(columns = ['Distance', 'CZ80', 'MSA 1993', 'MSA name', 'State place code'])

In [8]:
#merge cz level with migration data
cz_only = cz_map[["cz", "cz_pop", "pct_metro_pop", "urbanicity"]].drop_duplicates("cz")

o_czs = cz_only.add_prefix("o_")
d_czs = cz_only.add_prefix("d_")

race_inc = (race_inc.merge(o_czs, on= 'o_cz', how = 'left', validate = 'many_to_one')
            .merge(d_czs, on = 'd_cz', how = 'left', validate= 'many_to_one'))

In [9]:
#stayed is number of people who live in the same commuting zone at 16 ant 26
race_inc["stayed"] = race_inc["n"].where(race_inc["o_cz"] == race_inc["d_cz"], 0)

In [51]:
inc_race_matrix = (race_inc.groupby(["race", "income_quintile"], as_index=False)
           .agg(total=("n", "sum"), stayed=("stayed", "sum")))
inc_race_matrix["share_left"] = 1 - inc_race_matrix["stayed"] / inc_race_matrix["total"]

chart = alt.Chart(inc_race_matrix).mark_rect().encode(
    alt.X("race:N", title = 'Race', sort = ["Asian", "Black", "Hispanic", "White", "Other"]),
    alt.Y("income_quintile:O", title = 'Income Quintile'),
    alt.Color('share_left', scale= seq(LOSS), 
              legend = alt.Legend(title = None, format = '%'))
).properties(
    width = 200,
    height = 200,
    title = alt.Title("Share of 26-year-olds living outside the metro area they grew up in, by race and income",
          subtitle = "Roughly 31% of all 26-year-olds no longer live where they did when they were 16")
).configure_scale(bandPaddingInner=0.02)

chart.save("../static-viz/race-inc.svg")

In [12]:
inc_race_matrix.loc[inc_race_matrix['share_left'].idxmin()]

race               Hispanic
income_quintile           3
total               1006759
stayed               794684
share_left         0.210651
Name: 12, dtype: object

In [13]:
inc_race_matrix.loc[inc_race_matrix['share_left'].idxmax()]

race                  White
income_quintile           5
total               5500409
stayed              3353000
share_left         0.390409
Name: 24, dtype: object

In [14]:
inc_race_matrix.agg(total=("total", "sum"), stayed=("stayed", "sum"))

,total,stayed
total,31434476.0,NaN
stayed,NaN,21581400.0


In [15]:
1 - 21581400/31434476

0.31344807529160024

In [17]:
#matrix by urbanicity
inc_race_urbanicity = (race_inc.groupby(["race", "income_quintile", "o_urbanicity"], as_index=False)
           .agg(total=("n", "sum"), stayed=("stayed", "sum")))
inc_race_urbanicity["share_left"] = 1 - inc_race_urbanicity["stayed"] / inc_race_urbanicity["total"]

In [ ]:
#matrix by urbanicity
chart = alt.Chart(inc_race_urbanicity).mark_rect().encode(
    alt.X("race:N", title = 'Race', sort = ["Asian", "Black", "Hispanic", "White", "Other"]),
    alt.Y("income_quintile:O", title = 'Income Quintile'),
    alt.Color('share_left',scale= seq(LOSS), legend=None)
).facet(
    facet = alt.Facet("o_urbanicity",sort = ['Large metro', 'Mid-size metro', 'Small metro', 'Rural'], title = None),
    columns = 1
).properties(title = "Share by population of home metro area").configure_scale(bandPaddingInner=0.02)

chart.save("../static-viz/race-inc-size.svg")

In [ ]:
#matrix by region
inc_race_region = (race_inc.groupby(["race", "income_quintile", "o_region"], as_index=False)
           .agg(total=("n", "sum"), stayed=("stayed", "sum")))
inc_race_region["share_left"] = 1 - inc_race_region["stayed"] / inc_race_region["total"]

chart = alt.Chart(inc_race_region).mark_rect().encode(
    alt.X("race:N", title = 'Race', sort = ["Asian", "Black", "Hispanic", "White", "Other"]),
    alt.Y("income_quintile:O", title = 'Income Quintile'),
    alt.Color('share_left', scale= seq(LOSS))
).facet(
    facet = alt.Facet("o_region", title = None),
    columns = 1
).properties(title = "Share by home region").configure_scale(bandPaddingInner=0.02)

chart.save("../static-viz/race-inc-region.svg")


alt.FacetChart(...)

In [ ]:
#get net changes in young adult population
out_tot = agg.groupby("o_cz")["n"].sum().rename("n_tot_o")   # grew up in cz
in_tot  = agg.groupby("d_cz")["n"].sum().rename("n_tot_d")   # live in cz

net = pd.concat([out_tot, in_tot], axis=1).fillna(0).rename_axis("cz").reset_index()
net["net_change"] = net["n_tot_d"] - net["n_tot_o"]
net["pct_change"] = net["net_change"] / net["n_tot_o"]

moves = agg[agg["o_cz"] != agg["d_cz"]]
outflow = moves.groupby("o_cz")["n"].sum().rename("outflow") #same as n_tot_o - n
inflow  = moves.groupby("d_cz")["n"].sum().rename("inflow") #same as n_tot_d - n

net = net.merge(outflow, left_on="cz", right_index=True, how="left") \
         .merge(inflow, left_on="cz", right_index=True, how="left") \
         .fillna({"inflow": 0, "outflow": 0})

names = agg[["o_cz", "o_cz_name", "o_state_name"]].drop_duplicates("o_cz") \
           .rename(columns={"o_cz": "cz", "o_cz_name": "cz_name", "o_state_name": "state"})
net = net.merge(names, on="cz", how="left")

net = net.merge(cz_map, on="cz", how = 'left')

In [67]:
net['County FIPS Code'] = net['County FIPS Code'].astype(int)
counties = alt.topo_feature(data.us_10m.url, "counties")

county_map = alt.Chart(counties).mark_geoshape().transform_lookup(
    lookup='id',
    from_ = alt.LookupData(data = net, key = 'County FIPS Code', 
                           fields=['cz_pop', 'cz_name'])
    ).encode(
        color = alt.Color('cz_pop:Q',
                          scale = alt.Scale(range=[PAGE_BG, '#006d2c'],
                                             interpolate="lab"),
                          title = "Population"),              
    ).project(type='albersUsa').properties(width = 500, height = 500)

county_map.save("../static-viz/pop-us.svg")

In [ ]:
places = pd.DataFrame([
    {"name": "Clarksville, TN",      "lon": -87.35,  "lat": 36.53},
    {"name": "Manhattan, KS",  "lon": -96.57, "lat": 39.18},
    {"name": "Denver, CO",  "lon": -104.99, "lat": 39.73},
    {"name": "Jackson, WY",  "lon": -110.76, "lat": 43.48},
    {"name": "Fargo, ND",  "lon": 	-96.79, "lat": 46.87},
    {"name": "Austin, TX",       "lon": -97.74,  "lat": 30.27},
    {"name": "Bristol Bay, AK",  "lon": -157.02, "lat": 58.73}
])

county_map = alt.Chart(counties).mark_geoshape().transform_lookup(
    lookup='id',
    from_ = alt.LookupData(data = net, key = 'County FIPS Code', fields=['pct_change', 'cz_name'])
    ).encode(
        color = alt.Color('pct_change:Q',
                          scale = diverging(.7),
                          legend = alt.Legend(format = '%'),
                          title = None),
         tooltip=["cz_name:N", alt.Tooltip("pct_change:Q", format=".1%")]                  
    )

pts = alt.Chart(places).encode(longitude="lon:Q", latitude="lat:Q")

dots = pts.mark_circle(size=25, color=INK, opacity=0.9)
background = pts.mark_text(dy=-11, fontSize=11, fontWeight="bold",
                     stroke=PAGE_BG, strokeWidth=3, color=PAGE_BG).encode(text="name:N")
labels = pts.mark_text(dy=-11, fontSize=11, fontWeight="bold", color=INK).encode(text="name:N")

chart = (county_map + dots + background + labels
 ).project(type='albersUsa').properties(width = 500,
 height = 500, title = alt.Title("Percent change in population of people ages 16-26",
                                 subtitle = "Counties are colored by the commuting zone they belong to."))

chart.save("../static-viz/net-change-map.svg")

alt.LayerChart(...)

In [68]:
net['inflow_rate'] = net['inflow']/net['n_tot_o']
net['outflow_rate'] = net['outflow']/net['n_tot_o']

In [ ]:
base = alt.Chart(net).encode(
    x=alt.X("outflow_rate:Q", title="Outflow",
            axis=alt.Axis(format="%")),
    y=alt.Y("inflow_rate:Q", title="Inflow",
            axis=alt.Axis(format="%")),
)

points = base.mark_circle(opacity=0.4, color=INK_SOFT).encode(
    size=alt.Size("cz_pop:Q",scale=alt.Scale(range=[10, 1500]), legend = None)
)

label_dat = net[net['Name of largest place in commuting zone'].isin(['Austin city, TX', 'Clarksville city, TN', 'King Salmon CDP, AK', 
                                     'Fargo city, ND', 'Manhattan city, KS', 'Denver city, CO', 'Jackson town, WY'])]
label_dat = label_dat[['cz_name', 'outflow_rate', 'inflow_rate', 'state']].drop_duplicates()

labels = alt.Chart(label_dat).mark_text(
    dx = {"expr": "datum.outflow_rate < 0.4 ? -5 : 0"},
    dy = {"expr": "datum.inflow_rate < 0.4 ? 24 : -23"}
).transform_calculate(
    label="[datum.cz_name, datum.state]").encode(
    x = alt.X("outflow_rate:Q"),
    y=alt.Y("inflow_rate:Q"),
    text = "label:N"
    )

lim = max(net["outflow_rate"].max(), net["inflow_rate"].max()) * 1.05
sx = alt.Scale(domain=[0, lim], nice=False)
sy = alt.Scale(domain=[0, lim], nice=False)

# Gain triangle: between the diagonal and the top edge
gain_tri = alt.Chart(pd.DataFrame({"x": [0, lim], "y": [0, lim], "y2": [lim, lim]})).mark_area(
    color=GAIN_TINT, opacity=0.9
).encode(x=alt.X("x:Q", scale=sx), y=alt.Y("y:Q", scale=sy), y2="y2:Q")

# Loss triangle: between the bottom edge and the diagonal
loss_tri = alt.Chart(pd.DataFrame({"x": [0, lim], "y": [0, 0], "y2": [0, lim]})).mark_area(
    color=LOSS_TINT, opacity=0.9
).encode(x=alt.X("x:Q", scale=sx), y=alt.Y("y:Q", scale=sy), y2="y2:Q")


layered = alt.layer(gain_tri, loss_tri, points, labels).configure_axis(
    grid=False
)

chart = layered.properties(width = 600, height = 600, 
                           title = alt.Title("Young adults moving in and out, sized by total population",
                                             subtitle = "Inflow percentage of a metro area is measured as the number of people in the cohort who moved from elsewhere as a share of the number who grew up there." \
                                                                        "Outflow is the share of people who grew up there who now live somewhere else. Metro areas with higher inflow than outflow have a positive percentage change in population."),
                           )

chart.save("../static-viz/net-change-scatter.svg")


In [31]:
#add urbanicity and region to aggregated data
agg = (agg.merge(o_czs, on= 'o_cz', how = 'left', validate = 'many_to_one')
            .merge(d_czs, on = 'd_cz', how = 'left', validate= 'many_to_one'))

for side in ["o", "d"]:
    agg[f"{side}_division"] = agg[f"{side}_state_name"].map(state_to_division)
    agg[f"{side}_region"] = agg[f"{side}_state_name"].map(state_to_region)

In [32]:
region_sankey = agg[agg['o_cz'] != agg['d_cz']]
region_sankey = region_sankey.groupby(['o_region', 'd_region'], as_index=False).agg(n = ('n', 'sum'))

Sankey altair template attributed to https://anyplot.ai/sankey-basic/python/altair

In [33]:
# Layout
width = 620
height = 320
node_width = 18
node_padding = 6
fill_ratio = 0.84
top_margin = 40
bottom_margin = 34
label_margin = 90
available_height = height - top_margin - bottom_margin

# One color per region, used for source nodes, target nodes, and flows
region_colors = REGION
region_order = list(region_colors.keys())

sources = [r for r in region_order if r in set(region_sankey["o_region"])]
targets = [r for r in region_order if r in set(region_sankey["d_region"])]

source_totals = region_sankey.groupby("o_region")["n"].sum().to_dict()
target_totals = region_sankey.groupby("d_region")["n"].sum().to_dict()
total_flow = region_sankey["n"].sum()


def stack_nodes(names, totals):
    """Vertically stack nodes, centered, with padding between them."""
    heights = {k: totals[k] / total_flow * available_height * fill_ratio for k in names}
    total_h = sum(heights.values()) + node_padding * (len(names) - 1)
    y = top_margin + (available_height - total_h) / 2
    positions = {}
    for k in names:
        positions[k] = {"y": y, "height": heights[k]}
        y += heights[k] + node_padding
    return positions


source_positions = stack_nodes(sources, source_totals)
target_positions = stack_nodes(targets, target_totals)

# Node rectangles
nodes_data = []
for src in sources:
    pos = source_positions[src]
    nodes_data.append({
        "name": src, "x": 0, "x2": node_width,
        "y": pos["y"], "y2": pos["y"] + pos["height"],
        "color": region_colors[src],
        "label_x": - 8, "label_y": pos["y"] + pos["height"] / 2,
        "total": source_totals[src], "side": "source",
    })
for tgt in targets:
    pos = target_positions[tgt]
    nodes_data.append({
        "name": tgt, "x": width - node_width, "x2": width,
        "y": pos["y"], "y2": pos["y"] + pos["height"],
        "color": region_colors[tgt],                      # fixed: was source_colors[src]
        "label_x": width + 8, "label_y": pos["y"] + pos["height"] / 2,
        "total": target_totals[tgt], "side": "target",
    })
nodes_df = pd.DataFrame(nodes_data)

# Flow bands: smoothstep S-curves between source and target nodes
source_y_offsets = {s: source_positions[s]["y"] for s in sources}
target_y_offsets = {t: target_positions[t]["y"] for t in targets}

# Stack flows in a consistent order so bands don't cross unnecessarily
flows_sorted = region_sankey.assign(
    so=region_sankey["o_region"].map(region_order.index),
    to=region_sankey["d_region"].map(region_order.index),
).sort_values(["so", "to"])

all_flow_data = []
num_curve_points = 40
x_start, x_end = node_width, width - node_width

for _, row in flows_sorted.iterrows():
    src, tgt, val = row["o_region"], row["d_region"], row["n"]

    src_h = val / source_totals[src] * source_positions[src]["height"]
    tgt_h = val / target_totals[tgt] * target_positions[tgt]["height"]

    s_top, t_top = source_y_offsets[src], target_y_offsets[tgt]
    s_bot, t_bot = s_top + src_h, t_top + tgt_h
    source_y_offsets[src] += src_h
    target_y_offsets[tgt] += tgt_h

    def curve(y0, y1, reverse=False):
        idx = range(num_curve_points - 1, -1, -1) if reverse else range(num_curve_points)
        pts = []
        for i in idx:
            t = i / (num_curve_points - 1)
            s = t * t * (3 - 2 * t)
            pts.append((x_start + t * (x_end - x_start), y0 + s * (y1 - y0)))
        return pts

    polygon = curve(s_top, t_top) + curve(s_bot, t_bot, reverse=True)
    for i, (x, y) in enumerate(polygon):
        all_flow_data.append({"flow_id": f"{src}-{tgt}", "o_region": src, "d_region": tgt,
                              "n": val, "x": x, "y": y, "order": i})

flows_df = pd.DataFrame(all_flow_data)

xs = alt.Scale(domain=[-label_margin, width + label_margin])
ys = alt.Scale(domain=[0, height])

# Flows colored by source region
links_chart = (
    alt.Chart(flows_df)
    .mark_line(filled=True, opacity=0.55, strokeWidth=0)
    .encode(
        x=alt.X("x:Q", scale=xs, axis=None),
        y=alt.Y("y:Q", scale=ys, axis=None),
        color=alt.Color("o_region:N",
                        scale=alt.Scale(domain=region_order, range=list(region_colors.values())),
                        legend=None),
        detail="flow_id:N",
        order="order:Q",
        tooltip=[alt.Tooltip("o_region:N", title="From"),
                 alt.Tooltip("d_region:N", title="To"),
                 alt.Tooltip("n:Q", title="People", format=",")],
    )
)

# Node rectangles (color is passed through literally)
nodes_chart = (
    alt.Chart(nodes_df)
    .mark_rect(stroke="#444", strokeWidth=1)
    .encode(
        x=alt.X("x:Q", scale=xs, axis=None), x2="x2:Q",
        y=alt.Y("y:Q", scale=ys, axis=None), y2="y2:Q",
        color=alt.Color("color:N", scale=None),
        tooltip=[alt.Tooltip("name:N", title="Region"),
                 alt.Tooltip("total:Q", title="People", format=",")],
    )
)

def label_layer(side, align):
    return (
        alt.Chart(nodes_df[nodes_df["side"] == side])     
        .mark_text(fontSize=12, fontWeight="bold", align=align, baseline="middle", color="#222")
        .encode(x=alt.X("label_x:Q", scale=xs), y=alt.Y("label_y:Q", scale=ys), text="name:N")
    )

chart = (
    alt.layer(links_chart, nodes_chart, label_layer("source", "right"))
    .properties(width=width, height=height,
                title=alt.Title("Where people live now, by region they grew up in",
                                subtitle="Left: region of residence at 16. Right: region of residence at 26."))
    .configure_view(strokeWidth=0)
)
chart.save("../static-viz/region-sankey.svg")

In [34]:
urbanicity_sankey = agg[agg['o_cz'] != agg['d_cz']]
urbanicity_sankey = urbanicity_sankey.groupby(['o_urbanicity', 'd_urbanicity'], as_index=False).agg(n = ('n', 'sum'))

In [ ]:
# Layout
width = 620
height = 320
node_width = 18
node_padding = 6
fill_ratio = 0.84
top_margin = 40
bottom_margin = 34
label_margin = 90
available_height = height - top_margin - bottom_margin

# One color per urbanicity, used for source nodes, target nodes, and flows
urbanicity_colors = URBANICITY

urbanicity_order = list(urbanicity_colors.keys())

sources = [r for r in urbanicity_order if r in set(urbanicity_sankey["o_urbanicity"])]
targets = [r for r in urbanicity_order if r in set(urbanicity_sankey["d_urbanicity"])]

source_totals = urbanicity_sankey.groupby("o_urbanicity")["n"].sum().to_dict()
target_totals = urbanicity_sankey.groupby("d_urbanicity")["n"].sum().to_dict()
total_flow = urbanicity_sankey["n"].sum()


def stack_nodes(names, totals):
    """Vertically stack nodes, centered, with padding between them."""
    heights = {k: totals[k] / total_flow * available_height * fill_ratio for k in names}
    total_h = sum(heights.values()) + node_padding * (len(names) - 1)
    y = top_margin + (available_height - total_h) / 2
    positions = {}
    for k in names:
        positions[k] = {"y": y, "height": heights[k]}
        y += heights[k] + node_padding
    return positions


source_positions = stack_nodes(sources, source_totals)
target_positions = stack_nodes(targets, target_totals)

# Node rectangles
nodes_data = []
for src in sources:
    pos = source_positions[src]
    nodes_data.append({
        "name": src, "x": 0, "x2": node_width,
        "y": pos["y"], "y2": pos["y"] + pos["height"],
        "color": urbanicity_colors[src],
        "label_x": - 8, "label_y": pos["y"] + pos["height"] / 2,
        "total": source_totals[src], "side": "source",
    })
for tgt in targets:
    pos = target_positions[tgt]
    nodes_data.append({
        "name": tgt, "x": width - node_width, "x2": width,
        "y": pos["y"], "y2": pos["y"] + pos["height"],
        "color": urbanicity_colors[tgt],                      # fixed: was source_colors[src]
        "label_x": width + 8, "label_y": pos["y"] + pos["height"] / 2,
        "total": target_totals[tgt], "side": "target",
    })
nodes_df = pd.DataFrame(nodes_data)

# Flow bands: smoothstep S-curves between source and target nodes
source_y_offsets = {s: source_positions[s]["y"] for s in sources}
target_y_offsets = {t: target_positions[t]["y"] for t in targets}

# Stack flows in a consistent order so bands don't cross unnecessarily
flows_sorted = urbanicity_sankey.assign(
    so=urbanicity_sankey["o_urbanicity"].map(urbanicity_order.index),
    to=urbanicity_sankey["d_urbanicity"].map(urbanicity_order.index),
).sort_values(["so", "to"])

all_flow_data = []
num_curve_points = 40
x_start, x_end = node_width, width - node_width

for _, row in flows_sorted.iterrows():
    src, tgt, val = row["o_urbanicity"], row["d_urbanicity"], row["n"]

    src_h = val / source_totals[src] * source_positions[src]["height"]
    tgt_h = val / target_totals[tgt] * target_positions[tgt]["height"]

    s_top, t_top = source_y_offsets[src], target_y_offsets[tgt]
    s_bot, t_bot = s_top + src_h, t_top + tgt_h
    source_y_offsets[src] += src_h
    target_y_offsets[tgt] += tgt_h

    def curve(y0, y1, reverse=False):
        idx = range(num_curve_points - 1, -1, -1) if reverse else range(num_curve_points)
        pts = []
        for i in idx:
            t = i / (num_curve_points - 1)
            s = t * t * (3 - 2 * t)
            pts.append((x_start + t * (x_end - x_start), y0 + s * (y1 - y0)))
        return pts

    polygon = curve(s_top, t_top) + curve(s_bot, t_bot, reverse=True)
    for i, (x, y) in enumerate(polygon):
        all_flow_data.append({"flow_id": f"{src}-{tgt}", "o_urbanicity": src, "d_urbanicity": tgt,
                              "n": val, "x": x, "y": y, "order": i})

flows_df = pd.DataFrame(all_flow_data)

xs = alt.Scale(domain=[0, width])
ys = alt.Scale(domain=[0, height])

# Flows colored by source urbanicity
links_chart = (
    alt.Chart(flows_df)
    .mark_line(filled=True, opacity=0.65, strokeWidth=0)
    .encode(
        x=alt.X("x:Q", scale=xs, axis=None),
        y=alt.Y("y:Q", scale=ys, axis=None),
        color=alt.Color("o_urbanicity:N",
                        scale=alt.Scale(domain=urbanicity_order, range=list(urbanicity_colors.values())),
                        legend=None),
        detail="flow_id:N",
        order="order:Q",
        tooltip=[alt.Tooltip("o_urbanicity:N", title="From"),
                 alt.Tooltip("d_urbanicity:N", title="To"),
                 alt.Tooltip("n:Q", title="People", format=",")],
    )
)

# Node rectangles (color is passed through literally)
nodes_chart = (
    alt.Chart(nodes_df)
    .mark_rect(stroke="#444", strokeWidth=1)
    .encode(
        x=alt.X("x:Q", scale=xs, axis=None), x2="x2:Q",
        y=alt.Y("y:Q", scale=ys, axis=None), y2="y2:Q",
        color=alt.Color("color:N", scale=None),
        tooltip=[alt.Tooltip("name:N", title="urbanicity"),
                 alt.Tooltip("total:Q", title="People", format=",")],
    )
)

def label_layer(side, align):
    return (
        alt.Chart(nodes_df[nodes_df["side"] == side])     
        .mark_text(fontSize=12, fontWeight="bold", align=align, baseline="middle", color="#222")
        .encode(x=alt.X("label_x:Q", scale=xs), y=alt.Y("label_y:Q", scale=ys), text="name:N")
    )

chart = (
    alt.layer(links_chart, nodes_chart, label_layer("source", "right"))
    .properties(width=width, height=height,
                title=alt.Title("Where people live now, by urbanicity they grew up in",
                                subtitle="Left: urbanicity of residence at 16. Right: urbanicity of residence at 26."))
    .configure_view(strokeWidth=0)
)
chart.save("../static-viz/urbanicity-sankey.svg")